# Sunspot exploration
Let us start by importing the sunspot number dataset from the [Bruxelles Observatory](https://www.sidc.be/silso/datafiles) rather than from [Kaggle](https://www.kaggle.com/robervalt/sunspots) like most people do. ;)

The dataset we use is labelled "Monthly mean total sunspot number [1/1749 - now]" on the observatory Web site. We download it and store a local copy as "sunspot.csv", unless that file already exists from a previous run.

In [1]:
import pandas as pd
import requests
import os

In [2]:
if not os.path.exists('sunspot.csv'):
    url = 'https://www.sidc.be/silso/INFO/snmtotcsv.php'
    r = requests.get(url, allow_redirects=True)
    open('sunspot.csv', 'wb').write(r.content)

The dataset has no column titles, so we add them ourselves, following the [on-line documentation](https://www.sidc.be/SILSO/infosndtot) which says:

Column 1-2: Gregorian calendar date (Year, Month).  
Column 3: Date in fraction of year.  
Column 4: Monthly mean total sunspot number.  
Column 5: Monthly mean standard deviation of the input sunspot numbers.  
Column 6: Number of observations used to compute the monthly mean total sunspot number.  
Column 7: Definitive/provisional marker. '1' indicates that the value is definitive. '0' indicates that the value is still provisional.

Missing values are indicated by a value of -1

In [3]:
df = pd.read_csv('sunspot.csv', delimiter=";", 
                 names= ["Year", "Month", "Year_Fraction", "SunspotNumber", "SunspotNumber_sd",
                         "NumObservations", "Definitive?"])
df = df[df["Year"]<2024]

Pandas' summary of the dataset:

In [4]:
df.describe()

,Year,Month,Year_Fraction,SunspotNumber,SunspotNumber_sd,NumObservations,Definitive?
count,3300.000000,3300.000000,3300.000000,3300.000000,3300.000000,3300.000000,3300.0
mean,1886.000000,6.500000,1886.497992,81.773788,5.637939,117.089697,1.0
std,79.397168,3.452576,79.397664,67.666430,5.289855,249.490775,0.0
min,1749.000000,1.000000,1749.042000,0.000000,-1.000000,-1.000000,1.0
25%,1817.000000,3.750000,1817.769250,24.100000,-1.000000,-1.000000,1.0
50%,1886.000000,6.500000,1886.496500,67.550000,5.400000,30.000000,1.0
75%,1955.000000,9.250000,1955.225000,122.400000,9.425000,31.000000,1.0
max,2023.000000,12.000000,2023.958000,398.200000,29.400000,1587.000000,1.0


The first and last records:

In [6]:
df[df["SunspotNumber"] > 398]

,Year,Month,Year_Fraction,SunspotNumber,SunspotNumber_sd,NumObservations,Definitive?
352,1778,5,1778.371,398.2,-1.0,-1,1


In [ ]:
df.tail()

Let us plot this time series.

In [ ]:
df.plot(x='Year_Fraction', y='SunspotNumber')

Modeling this time series is an interesting exercise in itself. There is obviously a frequential aspect (a 11-years cycle, a 22-years cycle with some variation). Here is an example of a [time series decomposition in R](https://rpubs.com/danielkurnia/timeseries_sunspots).

All we attempt here is a rough estimation of the length of a cycle, without any pretension to scientific rigour.

From the plot, we see that all minima have a count below 20. We thus isolate the points near the minima:

In [ ]:
minima = df[df["SunspotNumber"] < 20].copy()
minima

A quick plot shows that this selection is reasonable, because it still shows cycles with the same frequency:

In [ ]:
minima.plot(x='Year_Fraction', y='SunspotNumber')

Next, we assign a number to each minimum. We consider that a series of points starts a new minimum if it is more than five years later than the previous point - the number of five being a visual estimate from the plot.

In [ ]:
minima["number"] = minima["Year_Fraction"].diff().gt(5).cumsum()
minima

We thus find 24 minima. Let's compute the mean Year_Fraction for each of them:

In [ ]:
grouped = minima.groupby("number")
grouped_mean = grouped.mean()
grouped_mean["Year_Fraction"]

The differences between successive points give the period of the sunspot number:

In [ ]:
grouped_mean["Year_Fraction"].diff()

There are large variations, especially among the early data which is probably the least reliable. Let's take the average over the last ten periods:

In [ ]:
grouped_mean.tail(10)['Year_Fraction'].diff().mean()

In [ ]:
df.iloc[df["SunspotNumber"].argmax()]